## 2. Quy mô: số dòng × số cột × dung lượng

In [2]:
import os
import pandas as pd

# Define TABLES based on the available CSV files
TABLES = [
    'products', 'orders', 'inventory', 'returns', 'reviews',
    'promotions', 'sample_submission', 'payments', 'web_traffic',
    'shipments', 'geography', 'customers', 'sales', 'order_items'
]

# Define path_of function to construct file paths
from pathlib import Path
# Tự tìm thư mục gốc repo (chứa data/raw). Máy khác/Colab: đặt biến môi trường DS317_DATA_DIR.
_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir()), None)
DATA_DIR = os.environ.get('DS317_DATA_DIR') or (str(_ROOT / 'data' / 'raw' / 'datathon-2026-round-1') if _ROOT else '/content/sample_data')
def path_of(table_name):
    return os.path.join(DATA_DIR, table_name + '.csv')

# Define OUT_DIR for output files
OUT_DIR = (str(_ROOT / 'outputs' / 'TuongVi') if _ROOT else '/content') + '/'
os.makedirs(OUT_DIR, exist_ok=True)

data, rows = {}, []
for t in TABLES:
    p = path_of(t)
    df = pd.read_csv(p, low_memory=False)
    data[t] = df
    b = os.path.getsize(p)
    rows.append({'bảng': t, 'file': os.path.basename(p), 'số dòng': len(df), 'số cột': df.shape[1],
                 'MB (1024²)': round(b/1024**2, 2), 'MB (10⁶)': round(b/1e6, 2)})
scale = pd.DataFrame(rows)
tot = {'bảng':'TỔNG','file':'','số dòng':scale['số dòng'].sum(),'số cột':scale['số cột'].sum(),
       'MB (1024²)':round(scale['MB (1024²)'].sum(),2),'MB (10⁶)':round(scale['MB (10⁶)'].sum(),2)}
scale = pd.concat([scale, pd.DataFrame([tot])], ignore_index=True)
scale.to_csv(OUT_DIR + 'quy_mo_14_bang.csv', index=False)
scale

,bảng,file,số dòng,số cột,MB (1024²),MB (10⁶)
0,products,products.csv,2412,8,0.19,0.20
1,orders,orders.csv,646945,8,43.83,45.96
2,inventory,inventory.csv,60247,17,5.41,5.67
3,returns,returns.csv,39939,7,2.18,2.28
4,reviews,reviews.csv,113551,7,6.48,6.79
5,promotions,promotions.csv,50,10,0.00,0.00
6,sample_submission,sample_submission.csv,548,3,0.02,0.02
7,payments,payments.csv,646945,4,17.53,18.38
8,web_traffic,web_traffic.csv,3652,7,0.20,0.21
9,shipments,shipments.csv,566067,4,18.84,19.76


# Đếm lại các con số được nêu trong docx

In [3]:
oi, o = data['order_items'], data['orders']
print('order_items dòng      :', f"{len(oi):,}")
print('số đơn (orders)       :', f"{o['order_id'].nunique():,}")
print('tổng số lượng (Σ qty) :', f"{oi['quantity'].sum():,}")
print('products (SKU)        :', f"{len(data['products']):,}")
print('inventory dòng / SP   :', f"{len(data['inventory']):,} / {data['inventory']['product_id'].nunique():,}")
print('inventory số mốc snapshot:', data['inventory']['snapshot_date'].nunique(),
      '|', data['inventory']['snapshot_date'].min(), '→', data['inventory']['snapshot_date'].max())
print('sales ngày            :', f"{len(data['sales']):,}", '|', data['sales']['Date'].min(), '→', data['sales']['Date'].max())
print('web_traffic ngày      :', f"{len(data['web_traffic']):,}", '|', data['web_traffic']['date'].min(), '→', data['web_traffic']['date'].max())
print('customers / reviews / shipments:', f"{len(data['customers']):,} / {len(data['reviews']):,} / {len(data['shipments']):,}")

order_items dòng      : 714,669
số đơn (orders)       : 646,945
tổng số lượng (Σ qty) : 3,213,143
products (SKU)        : 2,412
inventory dòng / SP   : 60,247 / 1,624
inventory số mốc snapshot: 126 | 2012-07-31 → 2022-12-31
sales ngày            : 3,833 | 2012-07-04 → 2022-12-31
web_traffic ngày      : 3,652 | 2013-01-01 → 2022-12-31
customers / reviews / shipments: 121,930 / 113,551 / 566,067


### Sơ đồ 14 bảng

# Mermaid trên draw.io

erDiagram
    customers ||--o{ orders : "đặt"
    geography ||--o{ customers : "zip"
    geography ||--o{ orders : "zip giao hàng"
    orders ||--|{ order_items : "gồm"
    orders ||--|| payments : "thanh toán"
    orders ||--o| shipments : "giao hàng"
    orders ||--o{ returns : "trả hàng"
    orders ||--o{ reviews : "được đánh giá"
    products ||--o{ order_items : "được bán"
    products ||--o{ inventory : "tồn kho tháng"
    products ||--o{ returns : "bị trả"
    products ||--o{ reviews : "được đánh giá"
    customers ||--o{ reviews : "viết"
    promotions |o--o{ order_items : "promo_id"
    promotions |o--o{ order_items : "promo_id_2"
    sales ||..o{ orders : "Date = order_date"
    sales ||..o| web_traffic : "Date = date"

    customers {
        int customer_id PK
        int zip FK
        string city
        date signup_date
        string gender
        string age_group
        string acquisition_channel
    }
    geography {
        int zip PK
        string city
        string region
        string district
    }
    products {
        int product_id PK
        string product_name
        string category
        string segment
        string size
        string color
        float price
        float cogs
    }
    promotions {
        string promo_id PK
        string promo_name
        string promo_type
        float discount_value
        date start_date
        date end_date
        string applicable_category
        string promo_channel
        int stackable_flag
        int min_order_value
    }
    orders {
        int order_id PK
        date order_date
        int customer_id FK
        int zip FK
        string order_status
        string payment_method
        string device_type
        string order_source
    }
    order_items {
        int order_id FK "khóa ghép, 16 cặp trùng"
        int product_id FK "khóa ghép"
        int quantity
        float unit_price
        float discount_amount
        string promo_id FK "có thể rỗng"
        string promo_id_2 FK "có thể rỗng"
    }
    payments {
        int order_id PK, FK
        string payment_method
        float payment_value
        int installments
    }
    shipments {
        int order_id PK, FK
        date ship_date
        date delivery_date
        float shipping_fee
    }
    returns {
        string return_id PK
        int order_id FK
        int product_id FK
        date return_date
        string return_reason
        int return_quantity
        float refund_amount
    }
    reviews {
        string review_id PK
        int order_id FK
        int product_id FK
        int customer_id FK
        date review_date
        int rating
        string review_title
    }
    inventory {
        date snapshot_date PK
        int product_id PK, FK
        int stock_on_hand
        int units_received
        int units_sold
        int stockout_days
        float days_of_supply
        float fill_rate
        int stockout_flag
        int overstock_flag
        int reorder_flag
        float sell_through_rate
        string product_name
        string category
        string segment
        int year
        int month
    }
    sales {
        date Date PK
        float Revenue
        float COGS
    }
    web_traffic {
        date date PK
        int sessions
        int unique_visitors
        int page_views
        float bounce_rate
        float avg_session_duration_sec
        string traffic_source
    }
    sample_submission {
        date Date PK
        float Revenue
        float COGS
    }

## 4. Tính riêng tư của `customers`

customers gồm 7 cột: customer_id, zip, city, signup_date, gender, age_group, acquisition_channel.

In [5]:
cust = data['customers']
print('Các cột:', list(cust.columns))
print('Cột trông giống tên/email/SĐT/địa chỉ:',
      [c for c in cust.columns if any(k in c.lower() for k in ['name','mail','phone','addr','street','tel'])])
prof = pd.DataFrame({'kiểu': cust.dtypes.astype(str), 'số giá trị khác nhau': cust.nunique(),
                     'thiếu': cust.isna().sum(), 'ví dụ': cust.iloc[0]})
display(prof)
print(cust['gender'].value_counts().to_dict())
print(cust['age_group'].value_counts().to_dict())
print(cust['acquisition_channel'].value_counts().to_dict())
print('signup_date:', cust['signup_date'].min(), '→', cust['signup_date'].max())
geo = data['geography']
print('customers.city khớp geography.city:', (cust.merge(geo, on='zip', suffixes=('','_g'))
                                              .eval('city == city_g')).mean())

Các cột: ['customer_id', 'zip', 'city', 'signup_date', 'gender', 'age_group', 'acquisition_channel']
Cột trông giống tên/email/SĐT/địa chỉ: []


,kiểu,số giá trị khác nhau,thiếu,ví dụ
customer_id,int64,121930,0,1
zip,int64,31491,0,15201
city,object,42,0,Hai Phong
signup_date,object,3941,0,2021-12-30
gender,object,3,0,Female
age_group,object,5,0,35-44
acquisition_channel,object,6,0,social_media


{'Female': 59640, 'Male': 57457, 'Non-binary': 4833}
{'25-34': 36342, '35-44': 31920, '45-54': 23172, '18-24': 17039, '55+': 13457}
{'organic_search': 36450, 'social_media': 24448, 'paid_search': 24285, 'email_campaign': 14674, 'referral': 12270, 'direct': 9803}
signup_date: 2012-01-17 → 2022-12-31
customers.city khớp geography.city: 1.0


In [6]:
# Nguy cơ nhận dạng lại: kích thước nhóm k khi kết hợp các cột bán định danh
def k_anon(cols):
    s = cust.groupby(cols).size().rename('k').reset_index()
    per = cust.merge(s, on=cols)['k']
    return {'tổ hợp cột': ' + '.join(cols), 'số nhóm': len(s), 'k nhỏ nhất': int(s['k'].min()),
            'bản ghi duy nhất (k=1)': int((per == 1).sum()), '% duy nhất': round((per == 1).mean()*100, 2),
            '% có k<5': round((per < 5).mean()*100, 2)}
priv = pd.DataFrame([k_anon(c) for c in [['city','gender','age_group'], ['zip'], ['zip','gender','age_group'],
                                        ['gender','age_group','signup_date'], ['zip','gender','age_group','signup_date']]])
priv.to_csv(OUT_DIR + 'rui_ro_nhan_dang_lai.csv', index=False)
priv

,tổ hợp cột,số nhóm,k nhỏ nhất,bản ghi duy nhất (k=1),% duy nhất,% có k<5
0,city + gender + age_group,630,2,0,0.00,0.04
1,zip,31491,1,322,0.26,64.23
2,zip + gender + age_group,103840,1,87380,71.66,100.00
3,gender + age_group + signup_date,36493,1,10701,8.78,46.52
4,zip + gender + age_group + signup_date,121921,1,121912,99.99,100.00


In [7]:
# Kiểm tra signup_date có đáng tin không: đơn đặt trước ngày đăng ký
m = data['orders'].merge(cust[['customer_id','signup_date']], on='customer_id')
before = pd.to_datetime(m['order_date']) < pd.to_datetime(m['signup_date'])
print(f'Đơn đặt trước ngày đăng ký: {before.sum():,} / {len(m):,} = {before.mean()*100:.1f}%')

Đơn đặt trước ngày đăng ký: 477,453 / 646,945 = 73.8%


## 5. Kiểm chứng nhanh PA2 — số lượng theo SKU × tuần
**Định nghĩa (khớp với 2.2 và Phần 3 của docx):**
- Tuần tính từ Thứ 2 đến Chủ nhật; mốc cắt là cuối ngày Chủ nhật.
- Dữ liệu bắt đầu Thứ 4 (2012-07-04) và kết thúc Thứ 7 (2022-12-31), nên **bỏ 2 tuần không đầy đủ** ở hai đầu. Còn tuần đầy đủ từ 2012-07-09 đến 2022-12-19 (kết thúc CN 2022-12-25).
- Biến mục tiêu PA2: Σ `quantity` theo (SKU, tuần), gán tuần theo `orders.order_date`.
- Bản chính tính **mọi trạng thái đơn** (khớp con số 3.213.143 đơn vị trong docx). Quy tắc đơn hủy/trả là việc của Văn Quyền; cell cuối chạy thử bản loại đơn `cancelled` để xem độ nhạy.
- "Đủ lịch sử" = có ít nhất 52 tuần kể từ tuần bán đầu tiên (để có lag 52 tuần), tính đến mốc cắt.
- Chia theo thời gian như 4.1: train đến hết 2020, validation 2021, test 2022 (theo năm của ngày Thứ 2 đầu tuần).

In [19]:
import pandas as pd
import numpy as np
import time

# ==========================================
# 0. TẠO VÀ CHUẨN BỊ DỮ LIỆU df_sku_weekly
# ==========================================
# Hợp nhất chi tiết đơn hàng và đơn hàng để có ngày mua
oi_temp = data['order_items'][['order_id', 'product_id', 'quantity']]
o_temp = data['orders'][['order_id', 'order_date']]
m_df = pd.merge(oi_temp, o_temp, on='order_id')
m_df['order_date'] = pd.to_datetime(m_df['order_date'])

# Tính ngày Thứ Hai đầu tuần
m_df['week_start'] = m_df['order_date'] - pd.to_timedelta(m_df['order_date'].dt.weekday, unit='D')
full_weeks = pd.date_range(start='2012-07-09', end='2022-12-25', freq='W-MON')
m_df = m_df[(m_df['week_start'] >= full_weeks.min()) & (m_df['week_start'] <= full_weeks.max())]

# Tạo ma trận đầy đủ SKU x Tuần
all_skus = sorted(data['products']['product_id'].unique())
full_idx = pd.MultiIndex.from_product([all_skus, full_weeks], names=['sku_id', 'year_week'])
df_sku_weekly = pd.DataFrame(index=full_idx).reset_index()

# Tổng hợp doanh số thực tế
sales_agg = m_df.groupby(['product_id', 'week_start'])['quantity'].sum().reset_index()
sales_agg.columns = ['sku_id', 'year_week', 'sales']

# Kết hợp doanh số thực tế vào bảng
df_sku_weekly = pd.merge(df_sku_weekly, sales_agg, on=['sku_id', 'year_week'], how='left').fillna(0)
df_sku_weekly['year'] = df_sku_weekly['year_week'].dt.year
df_sku_weekly['week_of_year'] = df_sku_weekly['year_week'].dt.isocalendar().week.astype(int)

# Tìm tuần bán đầu tiên của mỗi SKU
first_sell = df_sku_weekly[df_sku_weekly['sales'] > 0].groupby('sku_id')['year_week'].min().reset_index()
first_sell.columns = ['sku_id', 'first_sell_week']
df_sku_weekly = pd.merge(df_sku_weekly, first_sell, on='sku_id', how='left')
df_sku_weekly['first_sell_week'] = df_sku_weekly['first_sell_week'].fillna(pd.Timestamp('2099-12-31'))

# ==========================================
# 0. ĐỊNH NGHĨA HÀM TÍNH ĐỘ ĐO (WMAPE & MAE)
# ==========================================
def calculate_wmape_mae(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    total_actual = np.sum(y_true)
    wmape = np.sum(np.abs(y_true - y_pred)) / total_actual if total_actual != 0 else np.nan
    mae = np.mean(np.abs(y_true - y_pred))

    return wmape, mae

# ==========================================
# 1. BƯỚC (1): TÍNH % TUẦN BÁN = 0 THEO SKU
# ==========================================
# Lọc các tuần từ tuần bán đầu tiên của mỗi SKU đến hết 2022 (Full Lifecycle)
df_lifecycle = df_sku_weekly[df_sku_weekly['year_week'] >= df_sku_weekly['first_sell_week']].copy()

# (1.1) % tuần bán = 0 theo chu kỳ sống (Full Lifecycle -> 72.25%)
pct_zero_lifecycle = (df_lifecycle['sales'] == 0).mean() * 100

# (1.2) % tuần bán = 0 trên toàn ma trận thô (SKU x All Weeks -> 87.79%)
pct_zero_full_matrix = (df_sku_weekly['sales'] == 0).mean() * 100

print(f"--- BƯỚC 1: TẠO BIẾN MỤC TIÊU ---")
print(f"% tuần bán = 0 (Từ tuần bán đầu tiên đến hết 2022): {pct_zero_lifecycle:.2f}%")
print(f"% tuần bán = 0 (Toàn bộ ma trận thô SKU x Weeks): {pct_zero_full_matrix:.2f}%\n")


# ==========================================
# 2. BƯỚC (2): LỌC MẪU VÀ ĐẾM SỐ MẪU
# ==========================================
# Điều kiện lọc: SKU phải đủ lịch sử >= 52 tuần kể từ tuần bán đầu tiên
sku_history_count = df_lifecycle.groupby('sku_id')['year_week'].nunique()
qualified_skus = sku_history_count[sku_history_count >= 52].index

# Tập dữ liệu Validation năm 2021 sau khi lọc SKU đủ lịch sử
val_2021_sku = df_sku_weekly[
    (df_sku_weekly['sku_id'].isin(qualified_skus)) &
    (df_sku_weekly['year'] == 2021)
].copy()

n_qualified_skus = len(qualified_skus)
n_samples_val = len(val_2021_sku)

# % tuần bán = 0 riêng trên tập Validation 2021 (PA2 Validation -> 84.2%)
pct_zero_val_2021 = (val_2021_sku['sales'] == 0).mean() * 100

print(f"--- BƯỚC 2: SỐ MẪU SAU LỌC ---")
print(f"Số SKU đủ lịch sử (>= 52 tuần): {n_qualified_skus} SKU")
print(f"Số mẫu trên Validation 2021 (SKU x Tuần): {n_samples_val} mẫu")
print(f"% tuần bán = 0 riêng trên tập Validation 2021: {pct_zero_val_2021:.2f}%\n")


# ==========================================
# 3. BƯỚC (3): CHẠY BASELINE TRÊN VALIDATION 2021
# ==========================================
start_time = time.time()

# ------------------------------------------
# Baseline A: Seasonal Naive (Lấy cùng kỳ năm ngoái - lùi 52 tuần)
# ------------------------------------------
# Ghép giá trị tuần w-52 vào năm 2021
df_2020 = df_sku_weekly[df_sku_weekly['year'] == 2020][['sku_id', 'week_of_year', 'sales']].rename(
    columns={'sales': 'pred_snaive'}
)

val_2021_sku = val_2021_sku.merge(df_2020, on=['sku_id', 'week_of_year'], how='left')
val_2021_sku['pred_snaive'] = val_2021_sku['pred_snaive'].fillna(0) # Trám 0 nếu năm 2020 chưa bán

wmape_sn, mae_sn = calculate_wmape_mae(val_2021_sku['sales'], val_2021_sku['pred_snaive'])

# ------------------------------------------
# Baseline B: Mean 4 Weeks (Trung bình 4 tuần gần nhất)
# ------------------------------------------
# Sắp xếp theo SKU và thời gian để tính trung bình động 4 tuần
df_sorted = df_sku_weekly.sort_values(['sku_id', 'year_week']).copy()
df_sorted['pred_mean_4w'] = df_sorted.groupby('sku_id')['sales'].transform(
    lambda x: x.shift(1).rolling(window=4, min_periods=1).mean()
)

val_2021_with_mean = val_2021_sku.merge(
    df_sorted[['sku_id', 'year_week', 'pred_mean_4w']],
    on=['sku_id', 'year_week'],
    how='left'
)
val_2021_with_mean['pred_mean_4w'] = val_2021_with_mean['pred_mean_4w'].fillna(0)

wmape_m4, mae_m4 = calculate_wmape_mae(val_2021_with_mean['sales'], val_2021_with_mean['pred_mean_4w'])

execution_time = time.time() - start_time

print(f"--- BƯỚC 3: KẾT QUẢ BASELINE TRÊN VALIDATION 2021 ---")
print(f"1. Seasonal Naive  -> WMAPE: {wmape_sn:.2%}, MAE: {mae_sn:.4f}")
print(f"2. Mean 4 Weeks    -> WMAPE: {wmape_m4:.2%}, MAE: {mae_m4:.4f}")
print(f"Thời gian chạy Baseline: {execution_time:.2f} giây")

--- BƯỚC 1: TẠO BIẾN MỤC TIÊU ---
% tuần bán = 0 (Từ tuần bán đầu tiên đến hết 2022): 72.25%
% tuần bán = 0 (Toàn bộ ma trận thô SKU x Weeks): 87.79%

--- BƯỚC 2: SỐ MẪU SAU LỌC ---
Số SKU đủ lịch sử (>= 52 tuần): 1515 SKU
Số mẫu trên Validation 2021 (SKU x Tuần): 78780 mẫu
% tuần bán = 0 riêng trên tập Validation 2021: 83.62%

--- BƯỚC 3: KẾT QUẢ BASELINE TRÊN VALIDATION 2021 ---
1. Seasonal Naive  -> WMAPE: 107.33%, MAE: 2.2522
2. Mean 4 Weeks    -> WMAPE: 76.64%, MAE: 1.6081
Thời gian chạy Baseline: 1.06 giây
